In [ ]:
import sys
sys.path.append("/path/to/example/autocompute/static")
from g16_env import load_gaussian_env
load_gaussian_env()

In [ ]:

import resource
resource.setrlimit(resource.RLIMIT_STACK, (resource.RLIM_INFINITY, resource.RLIM_INFINITY)) 

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, AddHs, RemoveHs, RWMol
from rdkit.Chem.rdmolfiles import MolToSmiles
from openbabel import openbabel, pybel
import os
import re
import pandas as pd
import subprocess
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openpyxl import Workbook
import time
import shutil
import glob

In [ ]:
import os
os.environ['PATH'] += ':/path/to/example/static/Multiwfn_3.8_dev_bin_Linux'

In [ ]:


def calculate_charge_and_multiplicity(smiles):
    
    
    mol = Chem.MolFromSmiles(smiles)
    
    
    mol = Chem.AddHs(mol)
    
    
    charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    
    
    multiplicity = sum([atom.GetNumRadicalElectrons() for atom in mol.GetAtoms()]) + 1
    
    return charge, multiplicity

In [ ]:

def create_repeatunit_Gaussian_inputfile(df, smiles, name, nproc=10, mem=20):
    
    
    name_terminal_indices = {}
    name_non_terminal_indices = {}

    
    mol = Chem.MolFromSmiles(smiles)
    mol_with_h = AddHs(mol)

    
    terminal_indices = [atom.GetIdx() for atom in mol_with_h.GetAtoms() if atom.GetSymbol() == '*']

    
    non_terminal_indices = [atom.GetIdx() for atom in mol_with_h.GetAtoms() if atom.GetSymbol() != '*']

    
    rw_mol = RWMol(mol_with_h)

    
    for idx in terminal_indices:
        rw_mol.ReplaceAtom(idx, Chem.Atom(1))  
        rw_mol.GetAtomWithIdx(idx).SetNoImplicit(False)  

    
    rw_mol.UpdatePropertyCache(strict=False)

    
    new_mol = rw_mol.GetMol()

    
    new_mol_with_h = AddHs(new_mol)

    
    smiles_with_h = Chem.MolToSmiles(new_mol_with_h)

    
    AllChem.EmbedMolecule(new_mol_with_h, AllChem.ETKDG())

    
    xyz = Chem.MolToXYZBlock(new_mol_with_h)

    
    filename = name.replace(' ', '_') + '.gjf'

    
    with open(filename, 'w') as file:
        file.write(xyz)

    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*$')

    
    with open(filename, 'r') as file:
        lines = file.readlines()

    
    start_index = 2  

    
    atom_lines = [line for line in lines[start_index:] if atom_line_pattern.match(line)]

    
    coordinates = ''.join(atom_lines)

    
    
    current_dir = os.getcwd()
    chk_file = os.path.join(current_dir, filename.replace('.gjf', '.chk'))


    
    
    charge, multiplicity = calculate_charge_and_multiplicity(smiles_with_h)


    
    
    with open(filename, 'w') as f:
        f.write(f"%nprocshared={nproc}\n")
        f.write(f"%mem={mem}GB\n")
        f.write(f"%chk={chk_file}\n")
        f.write("# opt freq b3lyp/6-311g(d,p) em=gd3bj scale=0.9682 \n\n")
        f.write(f"{filename}\n\n")
        f.write(f"{charge} {multiplicity}\n")
        f.write(coordinates)
        
        
        f.write("\n\n")

    
    name_terminal_indices[name] = terminal_indices
    name_non_terminal_indices[name] = non_terminal_indices

    return name_terminal_indices, name_non_terminal_indices, charge

In [ ]:

def run_gaussian_optfreq(file, success_dir, failure_dir):
    
    
    
    base_name = os.path.splitext(file)[0]
    output_file = base_name + '.out'
    chk_file = base_name + '.chk'
    
    
    try:
        subprocess.run(['g16', file, output_file], check=True)
        
        
        with open(output_file, 'r') as f:
            content = f.read()
            
            if 'Normal termination' in content:
                
                os.rename(file, os.path.join(success_dir, file))
                os.rename(output_file, os.path.join(success_dir, output_file))
                os.rename(chk_file, os.path.join(success_dir, chk_file))
                return output_file, True
            else:
                
                os.rename(file, os.path.join(failure_dir, file))
                os.rename(output_file, os.path.join(failure_dir, output_file))
                os.rename(chk_file, os.path.join(failure_dir, chk_file))
    except subprocess.CalledProcessError:
        
        os.rename(file, os.path.join(failure_dir, file))
        if os.path.exists(output_file):
            os.rename(output_file, os.path.join(failure_dir, output_file))
        if os.path.exists(chk_file):
            os.rename(chk_file, os.path.join(failure_dir, chk_file))

    return None, False

In [ ]:

def calculate_repeatunit_RESP(filename):
    
    base_name = os.path.splitext(filename)[0]
    chgcons_file = base_name + '_chgcons' + '.txt'
    
    
    commands = f'7\n18\n6\n1\n{chgcons_file}\n2\ny\n0\n0\nq'
    
    
    log_file = os.path.join(os.getcwd(), "calculate_repeatunit_RESP.txt")

    
    with open(log_file, "w") as log:
        try:
            
            process = subprocess.Popen(
                ['/usr/local/bin/Multiwfn', filename],
                stdin=subprocess.PIPE,
                stdout=log,
                stderr=log,
                text=True
            )

            
            process.communicate(commands)
            
            
            if process.returncode != 0:
                log.write("Multiwfn encountered an error. Please check the output above for details.\n")
                raise Exception("Multiwfn execution failed. Check calculate_repeatunit_RESP.txt for details.")

        except Exception as e:
            
            log.write(f"An exception occurred: {str(e)}\n")
            raise

In [ ]:

def convert_chk_to_fchk(Gaussian_path):
    
    for filename in os.listdir(Gaussian_path):
        
        if filename.endswith('.chk'):
            
            base_filename = os.path.splitext(filename)[0]
            
            input_file = os.path.join(Gaussian_path, filename)
            
            output_file = base_filename + '.fchk'
            
            command = ['/root/Gaussian16_Linux_AVX2/tar/g16/formchk', input_file, output_file]
            
            try:
                
                subprocess.run(command, check=True)
                print(f"Converted {input_file} to {output_file}")
            except subprocess.CalledProcessError as e:
                print(f"Failed to convert {input_file}: {e}")

In [ ]:

def create_chgcons_files(non_terminal_indices, charge):
    
    for name, indices in non_terminal_indices.items():
        
        updated_indices = [index + 1 for index in indices]
        
        
        indices_str = ','.join(map(str, updated_indices))
        
        
        filename = f"{name}_chgcons.txt"
        
        
        with open(filename, 'w') as file:
            file.write(f"{indices_str} {charge}")

In [ ]:
def calculate(success_dir, failure_dir, MAX_TASKS=6):
    
    start_time = time.time()

    
    gjf_files = [f for f in os.listdir('.') if os.path.isfile(f) and f.endswith('.gjf')]

    
    with ThreadPoolExecutor(max_workers=MAX_TASKS) as executor:
        
        futures = [executor.submit(run_gaussian_optfreq, file, success_dir, failure_dir) for file in gjf_files] 
        
        
        results = [f.result() for f in futures]

    
    success_results = [result for result in results if result[1]]

    
    end_time = time.time()

    
    elapsed_time = end_time - start_time
    print(f"The code ran for {elapsed_time} seconds.")

In [ ]:

def create_repeatunit_chg(polymer_name):
    for filename in os.listdir('.'):
        
        if filename.endswith('.fchk'):
            
            for name in polymer_name:
                if filename.startswith(name):
                    
                    calculate_repeatunit_RESP(filename)

In [ ]:

def read_excel_and_create_lists(filepath):
    
    df = pd.read_excel(filepath)
    
    polymers = df['Name'].astype(str).tolist()
    
    return df, polymers

In [ ]:

def get_unique_filenames(directory):
    
    filenames = set(os.path.splitext(file)[0] for file in os.listdir(directory))
    return filenames

In [ ]:

def copy_files_for_polymers(source_directory, destination_directory, polymers_list):
    
    os.makedirs(destination_directory, exist_ok=True)
    
    unique_filenames = get_unique_filenames(source_directory)
    
    not_found_polymers = []
    found_polymers = []
    
    
    for polymer in polymers_list:
        if polymer in unique_filenames:
            found_polymers.append(polymer)
            for extension in ['.gjf', '.chk', '.out']:
                source_file = os.path.join(source_directory, polymer + extension)
                destination_file = os.path.join(destination_directory, polymer + extension)
                if os.path.exists(source_file):
                    try:
                        shutil.copyfile(source_file, destination_file)
                    except Exception as e:
                        print(f"Error copying {source_file}: {e}")
                else:
                    print(f"File not found: {source_file}")
            
            polymer_chgcons = f"{polymer}_chgcons.txt"
            
            source_file = os.path.join(source_directory, polymer_chgcons)
            destination_file = polymer_chgcons
            if os.path.exists(source_file):
                try:
                    shutil.copyfile(source_file, destination_file)
                except Exception as e:
                    print(f"Error copying {source_file}: {e}")
            else:
                print(f"File not found: {source_file}")
        else:
            not_found_polymers.append(polymer)
            
    
    print("Found polymers:", found_polymers)
    
    print("Polymers not found:", not_found_polymers)
    return not_found_polymers, found_polymers

In [ ]:

def copy_newly_calculated_files(RESPpolymer_database_path, success_directory, not_found_polymers):
    
    os.makedirs(RESPpolymer_database_path, exist_ok=True)
    
    
    success_filenames = get_unique_filenames(success_directory)

    
    for polymer in not_found_polymers:
        if polymer in success_filenames:
            for extension in ['.gjf', '.chk', '.out']:
                source_file = os.path.join(success_directory, polymer + extension)
                destination_file = os.path.join(RESPpolymer_database_path, polymer + extension)
                
                shutil.copyfile(source_file, destination_file)
                print("Public message removed for release.")
            
            polymer_chgcons = f"{polymer}_chgcons.txt"
            
            source_file = polymer_chgcons
            destination_file = os.path.join(RESPpolymer_database_path, polymer_chgcons)
            shutil.copyfile(source_file, destination_file)
            print("Public message removed for release.")

In [ ]:

def create_input_files_for_missing_polymers(df, missing_polymers):
    for name in missing_polymers:
        smiles = df[df['Name'] == name]['SMILES'].iloc[0]
        
        terminal_indices, non_terminal_indices, repeatunit_charge = create_repeatunit_Gaussian_inputfile(df, smiles, name)
        
        create_chgcons_files(non_terminal_indices, repeatunit_charge)
        print(f"--------{name}-------{name}----------{name}--------{name}---------{name}-------{name}----------{name}--------{name}--------")

In [ ]:

def check_and_create_gaussian_database():
    
    home_directory = '/data'
    gaussian_database_path = os.path.join(home_directory, 'Gaussian_database')
    
    optfreq_gaussian_database_path = os.path.join(home_directory, 'Gaussian_database/opt+freq')
    RESPpolymer_database_path = os.path.join(home_directory, 'Gaussian_database/RESPpolymer')
    
    
    if not os.path.exists(gaussian_database_path):
        
        os.makedirs(gaussian_database_path)
        os.makedirs(optfreq_gaussian_database_path)
        os.makedirs(RESPpolymer_database_path)
        print("Public message removed for release.")
        
    elif not os.path.exists(optfreq_gaussian_database_path) and os.path.exists(RESPpolymer_database_path):
        os.makedirs(optfreq_gaussian_database_path)
        print("Public message removed for release.")
        
    elif not os.path.exists(RESPpolymer_database_path) and os.path.exists(optfreq_gaussian_database_path):
        os.makedirs(RESPpolymer_database_path)
        print("Public message removed for release.")

    elif not os.path.exists(RESPpolymer_database_path) and not os.path.exists(optfreq_gaussian_database_path):
        os.makedirs(optfreq_gaussian_database_path)
        os.makedirs(RESPpolymer_database_path)
        print("Public message removed for release.")
        
    else:
        
        print("Public message removed for release.")
    return gaussian_database_path, optfreq_gaussian_database_path, RESPpolymer_database_path

In [ ]:


os.makedirs('Gaussian/RESPpolymer/success', exist_ok=True)
os.makedirs('Gaussian/RESPpolymer/failure', exist_ok=True)


base_dir = 'Gaussian/RESPpolymer'
success_dir = os.path.join(base_dir, 'success')
failure_dir = os.path.join(base_dir, 'failure')


Gaussian_path = "Gaussian/RESPpolymer/success"


gaussian_database_path, optfreq_gaussian_database_path, RESPpolymer_database_path = check_and_create_gaussian_database()


df, polymers = read_excel_and_create_lists('System_homopolymer.xlsx')


not_found_polymers, found_polymers = copy_files_for_polymers(
    RESPpolymer_database_path,
    success_dir,
    polymers
)

total_polymers = not_found_polymers + found_polymers


create_input_files_for_missing_polymers(df, not_found_polymers)



calculate(success_dir, failure_dir, MAX_TASKS=1)  


copy_newly_calculated_files(RESPpolymer_database_path, success_dir, not_found_polymers)


convert_chk_to_fchk(Gaussian_path)


with open("test1_pass.txt", "w") as f:
    f.write("convert_chk_to_fchk completed successfully.\n")

create_repeatunit_chg(total_polymers)


with open("test2_pass.txt", "w") as f:
    f.write("create_repeatunit_chg completed successfully.\n")
